# Telco Customer Churn — Model Selection, Model Analysis & Conclusions
### Part B of the statistical analysis project

**Reference text:** Agresti, A. & Kateri, M. — *Foundations of Statistics for Data Scientists: With R and Python* (CRC Press). The methods used here follow Chapter 7 (Generalized Linear Models: logistic regression, deviance, likelihood-ratio tests, AIC) and Chapter 8 (Classification: ROC curves, classification trees).

---

#### Business question
A telecom company loses about **27% of its customers**. We want to (1) understand *which customer characteristics are associated with churn*, (2) build a model that *predicts* who is likely to leave, and (3) recommend *which customers the retention team should target and why*.

#### Data
IBM Telco Customer Churn dataset — 7,043 customers, 21 variables:
- **Demographics:** gender, SeniorCitizen, Partner, Dependents
- **Account:** tenure (months), Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges
- **Services:** PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies
- **Response:** `Churn` (Yes/No) → modeled as a binary variable $y_i \in \{0,1\}$

#### Statistical model
Binary logistic regression (a GLM with binomial random component and logit link):

$$\operatorname{logit}\,P(y_i = 1) = \log\frac{\pi_i}{1-\pi_i} = \beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip}$$

$e^{\beta_j}$ is the **odds ratio**: the multiplicative change in the odds of churn for a one-unit change in $x_j$, holding the other predictors fixed. Interpretable coefficients are the reason logistic regression is the main model.

#### Plan of this notebook
| Step | Content |
|---|---|
| 1 | Load and clean the data |
| 2 | Stratified 80/20 train/test split (fixed seed) |
| 3 | Full logistic model |
| 4 | Multicollinearity diagnosis (VIF) |
| 5 | Model reduction: backward stepwise AIC + likelihood-ratio test |
| 6 | Linearity of the logit for tenure (empirical logits, Box-Tidwell) |
| 7 | Final model and term-by-term LR tests |
| 8 | Influential observations and goodness of fit |
| 9 | **Odds ratios with 95% confidence intervals** |
| 10 | Decision tree benchmark |
| 11 | Test-set comparison of all candidate models (ROC-AUC) |
| 12 | Class imbalance and threshold choice |
| 13 | Gains / lift |
| 14 | Recommendations: which customers to target |
| 15 | Conclusions |

> **Reproducibility:** every random step uses `SEED = 42`. The CSV must be in the same folder as this notebook (or change `DATA_PATH`).

In [16]:
# ============================================================
# Imports and global settings for the whole project
# Requirements: pip install pandas numpy scipy statsmodels scikit-learn plotly nbformat
# (optional, for exporting figures as PNG for the report: pip install kaleido)
# ============================================================
import re
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats

# Statistical modelling (GLMs, tests, diagnostics)
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Machine learning utilities (split, CV, tree benchmark, metrics)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score,
                             brier_score_loss, log_loss)

# Interactive, modern graphics
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from IPython.display import display, Markdown

# ---------- Global settings ----------
SEED = 42
DATA_PATH = Path("../../data") / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# ---------- Visual identity used in every chart ----------
PAL = {
    "churn": "#EF476F",   # churners / risk
    "stay":  "#118AB2",   # retained / protective
    "green": "#06D6A0",
    "gold":  "#FFD166",
    "dark":  "#073B4C",
    "grey":  "#94A3B8",
}
pio.templates["churn"] = go.layout.Template(
    layout=dict(
        font=dict(family="Inter, Segoe UI, Helvetica, Arial, sans-serif", size=13, color="#1F2937"),
        colorway=[PAL["stay"], PAL["churn"], PAL["green"], PAL["gold"], PAL["dark"], PAL["grey"]],
        title=dict(font=dict(size=18, color="#0F172A"), x=0.01, xanchor="left"),
        plot_bgcolor="white", paper_bgcolor="white",
        xaxis=dict(showgrid=False, linecolor="#CBD5E1", ticks="outside", tickcolor="#CBD5E1"),
        yaxis=dict(gridcolor="#EEF2F7", zeroline=False, linecolor="#CBD5E1"),
        legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", yanchor="bottom", y=1.02, x=0),
        margin=dict(l=60, r=30, t=90, b=50),
        hoverlabel=dict(font_size=12),
    )
)
pio.templates.default = "plotly_white+churn"

print("Environment ready ✔")

Environment ready ✔


## Step 1 — Load and clean the data

**What we do**
1. `TotalCharges` is stored as text. We convert it to numeric; **11 rows are blank**. All of them have `tenure = 0` (brand-new customers who have not been billed yet) and none churned, so dropping them does not bias the analysis.
2. The six internet add-ons (OnlineSecurity, TechSupport, …) have a third level, *"No internet service"*, that is an exact copy of `InternetService = No`. Keeping it would make the design matrix rank-deficient (perfect collinearity), so it is recoded to *"No"*. Same for `MultipleLines = "No phone service"`.
3. `SeniorCitizen` is coded 0/1 but is categorical, so it becomes *No/Yes*.
4. The response is coded $y = 1$ if `Churn = "Yes"`.

**Graphs:** overall class balance (this is the 73/27 imbalance that matters later) and the raw churn rate by the three variables that will turn out to be the strongest predictors.

In [17]:
raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape)

# 1) TotalCharges -> numeric
raw["TotalCharges"] = pd.to_numeric(raw["TotalCharges"].astype(str).str.strip().replace("", np.nan))
blank = raw[raw["TotalCharges"].isna()]
print(f"\nRows with blank TotalCharges: {len(blank)}")
display(blank[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

df = raw.dropna(subset=["TotalCharges"]).copy()

# 2) Structural duplicate levels -> "No"
ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]
for col in ADDONS:
    df[col] = df[col].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

# 3) SeniorCitizen as a category, 4) binary response
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["y"] = (df["Churn"] == "Yes").astype(int)

CATS = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines",
        "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
        "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"]
NUMS = ["tenure", "MonthlyCharges", "TotalCharges"]
SERVICES = ["PhoneService", "MultipleLines", "InternetService"] + ADDONS

print("\nClean shape:", df.shape)
print("Churn proportion:\n", df["Churn"].value_counts(normalize=True).round(4))

Raw shape: (7043, 21)

Rows with blank TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.5500,NaN,No
753,3115-CZMZD,0,20.2500,NaN,No
936,5709-LVOEQ,0,80.8500,NaN,No
1082,4367-NUYAO,0,25.7500,NaN,No
1340,1371-DWPAZ,0,56.0500,NaN,No
3331,7644-OMVMY,0,19.8500,NaN,No
3826,3213-VVOLG,0,25.3500,NaN,No
4380,2520-SGTTA,0,20.0000,NaN,No
5218,2923-ARZLG,0,19.7000,NaN,No
6670,4075-WKNIU,0,73.3500,NaN,No



Clean shape: (7032, 22)
Churn proportion:
 Churn
No    0.7342
Yes   0.2658
Name: proportion, dtype: float64


In [26]:
# --- Graph 1a: class balance (donut) ---
counts = df["Churn"].value_counts()

fig = go.Figure(
    go.Pie(
        labels=counts.index,
        values=counts.values,
        hole=0.62,
        marker=dict(colors=[PAL["stay"], PAL["churn"]]),
        textinfo="label+percent",
        textfont_size=14,
        sort=False
    )
)

fig.update_layout(
    title="Class balance: about 1 in 4 customers churned",
    showlegend=False,
    width=600,
    height=380,
    annotations=[
        dict(
            text=f"n = {len(df):,}",
            x=0.5,
            y=0.5,
            showarrow=False,
            font_size=18
        )
    ]
)

fig.show()


# --- Graph 1b: churn rate by Contract, InternetService, PaymentMethod ---
base_rate = df["y"].mean()

vars_ = ["Contract", "InternetService", "PaymentMethod"]

fig = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=vars_,
    vertical_spacing=0.12
)

for i, v in enumerate(vars_, start=1):

    rate = (
        df.groupby(v)["y"]
        .mean()
        .sort_values(ascending=False)
    )

    fig.add_bar(
        x=rate.index,
        y=rate.values,
        row=i,
        col=1,
        marker_color=[
            PAL["churn"] if r > base_rate else PAL["stay"]
            for r in rate.values
        ],
        text=[f"{r:.0%}" for r in rate.values],
        textposition="outside",
        showlegend=False
    )

    fig.add_hline(
        y=base_rate,
        line_dash="dot",
        line_color=PAL["grey"],
        row=i,
        col=1
    )


fig.update_yaxes(
    tickformat=".0%",
    range=[0, 0.55]
)

fig.update_layout(
    title=(
        f"Observed churn rate by key variables "
        f"(dotted line = overall {base_rate:.1%})"
    ),
    height=900,
    width=600,
)

fig.show()

## Step 2 — Train/test split (80/20, stratified, fixed seed)

**Why:** model selection and parameter estimation are done **only on the training set**. The test set is kept aside to measure honest out-of-sample performance (Agresti & Kateri discuss this in the context of evaluating classifiers, Ch. 8).

**Stratified** on `y` so both sets keep the same 26.6% churn rate; this matters with imbalanced classes because a random split could give a test set with noticeably more or fewer churners. `random_state = SEED` makes the split reproducible.

In [27]:
train, test = train_test_split(df, test_size=0.20, stratify=df["y"], random_state=SEED)

split_summary = pd.DataFrame({
    "n": [len(df), len(train), len(test)],
    "churners": [df.y.sum(), train.y.sum(), test.y.sum()],
    "churn rate": [df.y.mean(), train.y.mean(), test.y.mean()],
}, index=["Full data", "Train (80%)", "Test (20%)"])
display(split_summary)

fig = go.Figure(go.Bar(x=split_summary.index, y=split_summary["churn rate"],
                       marker_color=[PAL["dark"], PAL["stay"], PAL["green"]],
                       text=[f"{v:.2%}<br>n={n:,}" for v, n in zip(split_summary["churn rate"], split_summary["n"])],
                       textposition="outside"))
fig.update_layout(title="Stratification check: the churn rate is identical in every partition",
                  yaxis=dict(tickformat=".0%", range=[0, 0.35]), height=380, width=600)
fig.show()

,n,churners,churn rate
Full data,7032,1869,0.2658
Train (80%),5625,1495,0.2658
Test (20%),1407,374,0.2658


## Step 3 — Full logistic regression model (M1)

We fit the model with **all 19 predictors** by maximum likelihood (IRLS), using `statsmodels.GLM` with a binomial family, as in Ch. 7 of the book. Categorical predictors enter as indicator (dummy) variables; the first level alphabetically is the reference category.

**Key outputs:**
- **Deviance** $D = -2(\ell_{\text{model}} - \ell_{\text{saturated}})$; differences in deviance between nested models give likelihood-ratio statistics.
- **AIC** $= -2\ell + 2k$; lower is better, it penalizes the number of parameters $k$.

**What to look for:** coefficients with implausible signs and huge standard errors. These are a warning of multicollinearity, which we diagnose in Step 4.

Two helper functions are defined here and reused through the notebook.

In [28]:
BINOMIAL = sm.families.Binomial()

def fit_logit(terms, data=None):
    """Fit a logistic GLM y ~ terms on the training set (default)."""
    data = train if data is None else data
    return smf.glm("y ~ " + " + ".join(terms), data=data, family=BINOMIAL).fit()

def pretty(name):
    """Make statsmodels parameter names readable: C(Contract)[T.Two year] -> Contract: Two year."""
    name = name.replace("np.log(tenure + 1)", "log(tenure + 1)")
    m = re.match(r"C\((\w+)\)\[T\.(.+)\]", name)
    if m:
        return f"{m.group(1)}: {m.group(2)}"
    m = re.match(r"C\((\w+)\)$", name)
    return m.group(1) if m else name

FULL_TERMS = [f"C({c})" for c in CATS] + NUMS
m1 = fit_logit(FULL_TERMS)
print(m1.summary())
print(f"\nM1  AIC = {m1.aic:.2f}   Deviance = {m1.deviance:.2f}   parameters = {len(m1.params)}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:                 5625
Model:                            GLM   Df Residuals:                     5601
Model Family:                Binomial   Df Model:                           23
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2313.7
Date:                Mon, 05 Oct 2026   Deviance:                       4627.5
Time:                        18:38:15   Pearson chi2:                 6.45e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.2849
Covariance Type:            nonrobust                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [31]:
# --- Graph 3: M1 coefficients with 95% CI (log-odds scale) ---
ci = m1.conf_int()
coef = pd.DataFrame({"term": [pretty(t) for t in m1.params.index], "b": m1.params.values,
                     "lo": ci[0].values, "hi": ci[1].values}).iloc[1:]  # drop intercept
coef = coef.sort_values("b")
fig = go.Figure(go.Scatter(
    x=coef["b"], y=coef["term"], mode="markers",
    marker=dict(size=9, color=np.where(coef["b"] > 0, PAL["churn"], PAL["stay"])),
    error_x=dict(type="data", symmetric=False, array=coef["hi"] - coef["b"],
                 arrayminus=coef["b"] - coef["lo"], color=PAL["grey"], thickness=1.5)))
fig.add_vline(x=0, line_dash="dot", line_color=PAL["dark"])
fig.update_layout(title="M1 (full model): very wide intervals for the service and price variables",
                  xaxis_title="Coefficient (log-odds)", height=650, width=620)
fig.show()

## Step 4 — Multicollinearity: Variance Inflation Factors

The VIF of predictor $j$ is $\text{VIF}_j = 1/(1-R_j^2)$, where $R_j^2$ comes from regressing $x_j$ on all other predictors. It measures how much the variance of $\hat\beta_j$ is inflated by correlation with the other predictors. A common rule of thumb is that VIF > 5 is a concern and VIF > 10 is serious.

We also check two *structural* relations suspected from the variable definitions:
- Is `MonthlyCharges` simply the **price list** of the services the customer buys?
- Is `TotalCharges` ≈ `tenure × MonthlyCharges`?

If so, these variables carry no information beyond the others, and they make the individual coefficients uninterpretable.

In [32]:
def vif_table(model):
    X = pd.DataFrame(model.model.exog, columns=model.model.exog_names)
    v = [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
    return pd.Series(v, index=[pretty(c) for c in X.columns[1:]], name="VIF").sort_values(ascending=False)

vif_m1 = vif_table(m1)
display(vif_m1.to_frame().round(2))

r2_price = smf.ols("MonthlyCharges ~ " + " + ".join(f"C({s})" for s in SERVICES), data=train).fit().rsquared
r_total = np.corrcoef(train["TotalCharges"], train["tenure"] * train["MonthlyCharges"])[0, 1]
print(f"R² of MonthlyCharges ~ services          : {r2_price:.4f}")
print(f"corr(TotalCharges, tenure × MonthlyCharges): {r_total:.4f}")

,VIF
MonthlyCharges,870.2400
InternetService: Fiber optic,149.2800
InternetService: No,104.1300
PhoneService: Yes,35.0900
StreamingMovies: Yes,24.3000
StreamingTV: Yes,24.2600
TotalCharges,10.8300
tenure,7.5400
MultipleLines: Yes,7.3000
DeviceProtection: Yes,7.0300


R² of MonthlyCharges ~ services          : 0.9988
corr(TotalCharges, tenure × MonthlyCharges): 0.9996


In [34]:
# --- Graph 4a: VIF of the full model (log scale) ---
v = vif_m1.sort_values()
fig = go.Figure(go.Bar(x=v.values, y=v.index, orientation="h",
                       marker_color=[PAL["churn"] if x > 10 else (PAL["gold"] if x > 5 else PAL["stay"]) for x in v.values],
                       text=[f"{x:,.1f}" for x in v.values], textposition="outside"))
for thr, lab in [(5, "VIF = 5"), (10, "VIF = 10")]:
    fig.add_vline(x=thr, line_dash="dot", line_color=PAL["dark"], annotation_text=lab, annotation_position="top")
fig.update_layout(title="M1: severe multicollinearity (red = VIF > 10)",
                  xaxis=dict(type="log", title="VIF (log scale)"), height=650, width=620)
fig.show()

# --- Graph 4b: TotalCharges is tenure × MonthlyCharges ---
s = train.sample(1500, random_state=SEED)
fig = px.scatter(s, x=s["tenure"] * s["MonthlyCharges"], y="TotalCharges", color="Churn",
                 color_discrete_map={"No": PAL["stay"], "Yes": PAL["churn"]}, opacity=0.55,
                 labels={"x": "tenure × MonthlyCharges"})
fig.add_trace(go.Scatter(x=[0, 9000], y=[0, 9000], mode="lines", name="y = x",
                         line=dict(color=PAL["dark"], dash="dot")))
fig.update_layout(title=f"TotalCharges ≈ tenure × MonthlyCharges (r = {r_total:.4f})", height=450, width=620)
fig.show()

**Decision.** `MonthlyCharges` is almost perfectly determined by the services (R² ≈ 0.999), and `TotalCharges` is essentially tenure × MonthlyCharges. Both are **removed**. We keep the service indicators because they are interpretable and actionable (a manager can offer *tech support*; nobody can "offer a lower MonthlyCharges" independently of the services).

*Interpretation caveat to remember:* after removing the price, the coefficients of the expensive services (fiber, streaming, multiple lines) partly absorb a "higher bill" effect.

## Step 5 — Model reduction: backward stepwise AIC and likelihood-ratio test

- **M2 (clean full):** all categorical predictors + `tenure`, without the two charge variables.
- **M3 (reduced):** backward elimination from M2. At each step we remove the term whose removal **lowers AIC the most**, and stop when no removal lowers AIC.

M3 is nested in M2, so we test $H_0$: *the removed coefficients are all 0* with the **likelihood-ratio statistic**

$$G^2 = D_{M3} - D_{M2} \;\sim\; \chi^2_{df}, \qquad df = \#\text{params}(M2) - \#\text{params}(M3).$$

A large p-value means the simpler model fits as well as the bigger one, so we prefer it (parsimony).

In [35]:
def lrt(reduced, full):
    """Likelihood-ratio test for nested GLMs: returns G², df, p-value."""
    g2 = reduced.deviance - full.deviance
    df_ = int(round(full.df_model - reduced.df_model))
    return g2, df_, stats.chi2.sf(g2, df_)

def backward_aic(terms):
    current = list(terms)
    model = fit_logit(current)
    path = [("Start (M2)", model.aic)]
    while True:
        candidates = {t: fit_logit([x for x in current if x != t]) for t in current}
        best = min(candidates, key=lambda t: candidates[t].aic)
        if candidates[best].aic < model.aic:
            current.remove(best)
            model = candidates[best]
            path.append((f"− {pretty(best)}", model.aic))
        else:
            break
    return current, model, pd.DataFrame(path, columns=["step", "AIC"])

CLEAN_TERMS = [f"C({c})" for c in CATS] + ["tenure"]
m2 = fit_logit(CLEAN_TERMS)
red_terms, m3, aic_path = backward_aic(CLEAN_TERMS)

display(aic_path.round(2))
print("Terms kept in M3:", [pretty(t) for t in red_terms])
g2, d, p = lrt(m3, m2)
print(f"\nLRT M3 vs M2:  G² = {g2:.3f},  df = {d},  p = {p:.3f}")

,step,AIC
0,Start (M2),"4,691.8000"
1,− Partner,"4,689.9000"
2,− gender,"4,688.0000"
3,− DeviceProtection,"4,686.3200"


Terms kept in M3: ['SeniorCitizen', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure']

LRT M3 vs M2:  G² = 0.512,  df = 3,  p = 0.916


In [36]:
# --- Graph 5: AIC path of backward elimination ---
fig = go.Figure(go.Scatter(x=aic_path["step"], y=aic_path["AIC"], mode="lines+markers+text",
                           text=[f"{a:.1f}" for a in aic_path["AIC"]], textposition="top center",
                           line=dict(color=PAL["stay"], width=3), marker=dict(size=11, color=PAL["dark"])))
fig.update_layout(title=f"Backward elimination by AIC (LRT M3 vs M2: G² = {g2:.2f}, df = {d}, p = {p:.2f})",
                  yaxis_title="AIC", height=400, width=620)
fig.show()

## Step 6 — Linearity of the logit for the continuous predictor (tenure)

Logistic regression assumes the **logit is linear** in each continuous predictor. We check it three ways:

1. **Empirical logits:** group tenure into bands and compute $\log\big(\bar y/(1-\bar y)\big)$ in each band. If the assumption holds, these points lie on a straight line.
2. **Box-Tidwell test:** add the term $x\log x$ (with $x = \text{tenure}+1$ to avoid $\log 0$). A significant coefficient means the linear form is inadequate.
3. **Compare functional forms by AIC:** linear vs quadratic vs $\log(\text{tenure}+1)$. These models are *not nested*, so we use AIC rather than an LRT.

In [37]:
# 1) Empirical logits by tenure band
bins = [-1, 3, 6, 12, 24, 36, 48, 60, 72]
t = train.assign(band=pd.cut(train["tenure"], bins))
emp = t.groupby("band", observed=True).agg(rate=("y", "mean"), n=("y", "size"), mid=("tenure", "mean"))
emp["logit"] = np.log(emp["rate"] / (1 - emp["rate"]))
display(emp.round(3))

# 2) Box-Tidwell
other = [x for x in red_terms if x != "tenure"]
bt_data = train.assign(tp=train["tenure"] + 1)
bt_data["tp_log_tp"] = bt_data["tp"] * np.log(bt_data["tp"])
bt = fit_logit(other + ["tp", "tp_log_tp"], data=bt_data)
print(f"Box-Tidwell term: coef = {bt.params['tp_log_tp']:.4f},  p = {bt.pvalues['tp_log_tp']:.2e}")

# 3) Functional forms
forms = {"linear: tenure": ["tenure"],
         "quadratic: tenure + tenure²": ["tenure", "I(tenure**2)"],
         "log(tenure + 1)": ["np.log(tenure + 1)"]}
form_aic = pd.Series({k: fit_logit(other + f).aic for k, f in forms.items()}, name="AIC")
display(form_aic.to_frame().round(2))

,rate,n,mid,logit
band,,,,
"(-1, 3]",0.5690,823,1.5940,0.2760
"(3, 6]",0.4520,325,4.8150,-0.1910
"(6, 12]",0.3700,576,9.4740,-0.5330
"(12, 24]",0.2850,818,18.3090,-0.9210
"(24, 36]",0.2040,678,30.2510,-1.3640
"(36, 48]",0.1980,601,42.3940,-1.3990
"(48, 60]",0.1490,670,54.5280,-1.7400
"(60, 72]",0.0680,1134,68.1910,-2.6190


Box-Tidwell term: coef = 0.0451,  p = 2.98e-19


,AIC
linear: tenure,"4,686.3200"
quadratic: tenure + tenure²,"4,628.4600"
log(tenure + 1),"4,587.2900"


In [41]:
# --- Graph 6: empirical logits vs the linear and log fits (univariate, for visualisation) ---
grid = pd.DataFrame({"tenure": np.arange(0, 73)})
lin = smf.glm("y ~ tenure", train, family=BINOMIAL).fit()
logm = smf.glm("y ~ np.log(tenure + 1)", train, family=BINOMIAL).fit()
to_logit = lambda p_: np.log(p_ / (1 - p_))

fig = go.Figure()
fig.add_trace(go.Scatter(x=emp["mid"], y=emp["logit"], mode="markers", name="Empirical logit (band)",
                         marker=dict(size=np.sqrt(emp["n"]) * 1.2, color=PAL["churn"], opacity=0.85,
                                     line=dict(color="white", width=1)),
                         hovertemplate="mean tenure %{x:.1f}<br>logit %{y:.2f}<extra></extra>"))
fig.add_trace(go.Scatter(x=grid["tenure"], y=to_logit(lin.predict(grid)), mode="lines",
                         name="Linear in tenure", line=dict(color=PAL["grey"], dash="dash", width=3)))
fig.add_trace(go.Scatter(x=grid["tenure"], y=to_logit(logm.predict(grid)), mode="lines",
                         name="Linear in log(tenure + 1)", line=dict(color=PAL["stay"], width=3)))
fig.update_layout(title="The logit drops steeply in the first year: log(tenure + 1) captures the curvature",
                  xaxis_title="Tenure (months)", yaxis_title="logit P(churn)", height=800, width=620)
fig.show()

**Decision.** The empirical logits are clearly concave, the Box-Tidwell term is highly significant, and $\log(\text{tenure}+1)$ has the lowest AIC by a wide margin. We replace `tenure` with `log(tenure + 1)`. Interpretation: **the first months of a customer's life matter much more than later months.**

## Step 7 — Final model (M4) and term-by-term likelihood-ratio tests

**M4** = the predictors of M3 with `log(tenure + 1)` instead of `tenure`.

We report:
- **Overall LRT** against the null (intercept-only) model: does the model explain churn at all?
- **Term-by-term LRTs** (drop one term at a time, analogous to a Type-II analysis of deviance). Categorical variables with several levels are tested *as a whole* (df = number of levels − 1). This is better than reading individual Wald p-values.
- **McFadden pseudo-R²** $= 1 - \ell_{M4}/\ell_{\text{null}}$.
- **VIF after the fix**, to confirm that multicollinearity has disappeared.

In [42]:
FINAL_TERMS = [x for x in red_terms if x != "tenure"] + ["np.log(tenure + 1)"]
m4 = fit_logit(FINAL_TERMS)
print(m4.summary())

null = fit_logit(["1"])
g2_all, df_all, p_all = lrt(null, m4)
print(f"\nOverall LRT vs null: G² = {g2_all:.1f}, df = {df_all}, p = {p_all:.2e}")
print(f"McFadden pseudo-R² = {1 - m4.llf / null.llf:.3f}")
print(f"AIC: M1 = {m1.aic:.1f} | M2 = {m2.aic:.1f} | M3 = {m3.aic:.1f} | M4 = {m4.aic:.1f}")

rows = []
for term in FINAL_TERMS:
    g2_t, df_t, p_t = lrt(fit_logit([x for x in FINAL_TERMS if x != term]), m4)
    rows.append({"term": pretty(term), "G²": g2_t, "df": df_t, "p-value": p_t})
lr_table = pd.DataFrame(rows).sort_values("G²", ascending=False).reset_index(drop=True)
display(lr_table.style.format({"G²": "{:.2f}", "p-value": "{:.2e}"}))

vif_m4 = vif_table(m4)
print(f"Max VIF in M4: {vif_m4.max():.2f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:                 5625
Model:                            GLM   Df Residuals:                     5606
Model Family:                Binomial   Df Model:                           18
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2274.6
Date:                Mon, 05 Oct 2026   Deviance:                       4549.3
Time:                        18:58:56   Pearson chi2:                 5.77e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.2948
Covariance Type:            nonrobust                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

,term,G²,df,p-value
0,log(tenure + 1),273.22,1,2.25e-61
1,InternetService,166.91,2,5.69e-37
2,Contract,111.26,2,6.91e-25
3,MultipleLines,22.53,1,2.07e-06
4,PhoneService,15.96,1,6.47e-05
5,PaymentMethod,15.81,3,1.24e-03
6,StreamingTV,15.00,1,1.08e-04
7,OnlineSecurity,14.06,1,1.77e-04
8,StreamingMovies,13.10,1,2.96e-04
9,PaperlessBilling,12.27,1,4.60e-04


Max VIF in M4: 2.63


In [44]:
# --- Graph 7a: importance of each term by its LR statistic ---
lt = lr_table.sort_values("G²")
fig = go.Figure(go.Bar(x=lt["G²"], y=lt["term"], orientation="h",
                       marker_color=[PAL["stay"] if p_ < 0.05 else PAL["grey"] for p_ in lt["p-value"]],
                       text=[f"G²={g:.1f}  (p={p_:.1e})" for g, p_ in zip(lt["G²"], lt["p-value"])],
                       textposition="outside"))
fig.update_layout(title="M4: contribution of each term (likelihood-ratio G²; grey = not significant at 5%)",
                  xaxis=dict(type="log", title="G² (log scale)"), height=520, width=620)
fig.show()

# --- Graph 7b: VIF before vs after ---
cmp = pd.DataFrame({"M1 (full)": vif_m1, "M4 (final)": vif_m4}).dropna().sort_values("M1 (full)")
fig = go.Figure()
fig.add_bar(y=cmp.index, x=cmp["M1 (full)"], name="M1 (full)", orientation="h", marker_color=PAL["churn"])
fig.add_bar(y=cmp.index, x=cmp["M4 (final)"], name="M4 (final)", orientation="h", marker_color=PAL["stay"])
fig.add_vline(x=10, line_dash="dot", line_color=PAL["dark"])
fig.update_layout(title="VIF before and after removing the charge variables", barmode="group",
                  xaxis=dict(type="log", title="VIF (log scale)"), height=600, width=620)
fig.show()

## Step 8 — Influential observations and goodness of fit

**Influence diagnostics** (from the GLM hat matrix):
- **Leverage** $h_{ii}$: how unusual an observation's covariate pattern is. Flag if $h_{ii} > 2p/n$.
- **Standardized (Pearson) residuals**: flag if $|r_i| > 3$.
- **Cook's distance** $D_i$: overall effect of deleting observation $i$ on all $\hat\beta$. Values > 1 are influential; $4/n$ is a very strict screening cut-off.

**Sensitivity check:** refit without the observations with $D_i > 4/n$ and compare the coefficients. This shows whether "cleaning" influential points is a good idea.

**Goodness of fit — Hosmer-Lemeshow test:** group observations into 10 deciles of predicted probability and compare observed vs expected churners with a $\chi^2_8$ statistic. A large p-value means no evidence of lack of fit. We also show it as a **calibration plot**.

In [45]:
infl = m4.get_influence().summary_frame()
n, k = len(train), len(m4.params)
infl = infl.join(train[["tenure", "Contract", "InternetService", "MonthlyCharges", "Churn"]])

print(f"Max Cook's D          : {infl['cooks_d'].max():.4f}   (# > 1: {(infl['cooks_d'] > 1).sum()})")
print(f"# Cook's D > 4/n      : {(infl['cooks_d'] > 4 / n).sum()}")
print(f"# leverage > 2p/n     : {(infl['hat_diag'] > 2 * k / n).sum()}   (max = {infl['hat_diag'].max():.4f})")
print(f"# |std. residual| > 3 : {(infl['standard_resid'].abs() > 3).sum()}")
print("\nTop 5 by Cook's distance:")
display(infl.sort_values("cooks_d", ascending=False).head(5)[
    ["tenure", "Contract", "InternetService", "MonthlyCharges", "Churn", "cooks_d", "hat_diag", "standard_resid"]])

# Sensitivity: drop D > 4/n and refit
keep_idx = infl.index[infl["cooks_d"] <= 4 / n]
m4_trim = fit_logit(FINAL_TERMS, data=train.loc[keep_idx])
sens = pd.DataFrame({"All data": m4.params, "Without D > 4/n": m4_trim.params})
sens.index = [pretty(i) for i in sens.index]
display(sens.round(3))

Max Cook's D          : 0.0032   (# > 1: 0)
# Cook's D > 4/n      : 339
# leverage > 2p/n     : 288   (max = 0.0118)
# |std. residual| > 3 : 88

Top 5 by Cook's distance:


,tenure,Contract,InternetService,MonthlyCharges,Churn,cooks_d,hat_diag,standard_resid
3779,66,Two year,DSL,79.4000,Yes,0.0032,0.0008,8.6353
6724,64,Two year,DSL,70.2000,Yes,0.0030,0.0008,8.2470
6813,64,Two year,DSL,47.8500,Yes,0.0030,0.0014,6.4727
4513,72,Two year,DSL,92.4500,Yes,0.0029,0.0010,7.5698
4272,56,Two year,No,25.1500,Yes,0.0029,0.0008,8.5873


,All data,Without D > 4/n
Intercept,0.9060,1.1380
SeniorCitizen: Yes,0.2020,0.2150
Dependents: Yes,-0.2010,-0.3460
PhoneService: Yes,-0.5890,-0.8080
MultipleLines: Yes,0.4280,0.8060
InternetService: Fiber optic,0.9430,1.5820
InternetService: No,-0.8300,-1.0870
OnlineSecurity: Yes,-0.3550,-0.8850
OnlineBackup: Yes,-0.1370,-0.1660
TechSupport: Yes,-0.2880,-0.5080


In [46]:
# Hosmer-Lemeshow test
def hosmer_lemeshow(y, p_hat, g=10):
    d_ = pd.DataFrame({"y": y, "p": p_hat})
    d_["group"] = pd.qcut(d_["p"], g, labels=False, duplicates="drop")
    tab = d_.groupby("group").agg(obs=("y", "sum"), exp=("p", "sum"), n=("y", "size"), mean_p=("p", "mean"))
    hl = (((tab["obs"] - tab["exp"]) ** 2) / (tab["exp"] * (1 - tab["exp"] / tab["n"]))).sum()
    return hl, stats.chi2.sf(hl, g - 2), tab

hl, hl_p, hl_tab = hosmer_lemeshow(train["y"].values, m4.fittedvalues.values)
print(f"Hosmer-Lemeshow: χ² = {hl:.2f}, df = 8, p = {hl_p:.3f}")

Hosmer-Lemeshow: χ² = 7.02, df = 8, p = 0.535


In [47]:
# --- Graph 8a: influence plot (leverage vs residual, bubble = Cook's D) ---
fig = px.scatter(infl.reset_index(), x="hat_diag", y="standard_resid", size="cooks_d", color="Churn",
                 color_discrete_map={"No": PAL["stay"], "Yes": PAL["churn"]}, size_max=22, opacity=0.6,
                 hover_data=["tenure", "Contract", "InternetService", "cooks_d"],
                 labels={"hat_diag": "Leverage h_ii", "standard_resid": "Standardized residual"})
fig.add_hline(y=3, line_dash="dot", line_color=PAL["dark"]); fig.add_hline(y=-3, line_dash="dot", line_color=PAL["dark"])
fig.add_vline(x=2 * k / n, line_dash="dot", line_color=PAL["gold"], annotation_text="2p/n")
fig.update_layout(title=f"Influence plot: no observation is influential (max Cook's D = {infl['cooks_d'].max():.4f})",
                  height=480, width=620)
fig.show()

# --- Graph 8b: calibration plot (Hosmer-Lemeshow groups) ---
hl_tab["obs_rate"] = hl_tab["obs"] / hl_tab["n"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect calibration",
                         line=dict(color=PAL["grey"], dash="dot")))
fig.add_trace(go.Scatter(x=hl_tab["mean_p"], y=hl_tab["obs_rate"], mode="lines+markers", name="M4 (deciles)",
                         marker=dict(size=11, color=PAL["churn"]), line=dict(color=PAL["churn"], width=3)))
fig.update_layout(title=f"Calibration: predicted vs observed churn (Hosmer-Lemeshow p = {hl_p:.2f})",
                  xaxis=dict(title="Mean predicted probability", tickformat=".0%"),
                  yaxis=dict(title="Observed churn rate", tickformat=".0%"), height=460, width=620)
fig.show()

**Decision.** No single observation is influential. The large residuals are long-tenure, two-year-contract customers who churned anyway: surprising cases, not data errors. Deleting all points with $D_i > 4/n$ removes exactly those churners and causes **quasi-complete separation** (the two-year contract coefficient explodes towards −∞). So **we keep all observations**, and the model shows no evidence of lack of fit.

## Step 9 — Odds ratios with 95% confidence intervals (heart of the report)

For each predictor, $\widehat{OR}_j = e^{\hat\beta_j}$ and the 95% CI is $\exp\big(\hat\beta_j \pm 1.96\,SE(\hat\beta_j)\big)$ (Wald interval).

> The book recommends **profile-likelihood** intervals, which are more accurate when $n$ is small or estimates are near the boundary. With $n = 5{,}625$ they are practically identical to Wald intervals, so we report Wald and mention this in the report.

**How to read them** (each OR is *adjusted* for every other variable in the model):
- OR > 1 → higher odds of churn than the reference category; OR < 1 → lower (protective).
- A CI that contains 1 → no significant association at the 5% level.
- **log(tenure + 1):** because the predictor is on a log scale, the clearest summary is the effect of **doubling** tenure: $OR_{\times 2} = 2^{\hat\beta}$.

**Caveat:** these are *associations* in observational data, not causal effects.

In [49]:
ci4 = m4.conf_int()
or_table = pd.DataFrame({
    "OR": np.exp(m4.params), "CI 2.5%": np.exp(ci4[0]), "CI 97.5%": np.exp(ci4[1]), "p-value": m4.pvalues,
}).drop("Intercept")
or_table.index = [pretty(i) for i in or_table.index]
or_table["% change in odds"] = (or_table["OR"] - 1) * 100
or_table = or_table.sort_values("OR")
display(or_table.style.format({"OR": "{:.3f}", "CI 2.5%": "{:.3f}", "CI 97.5%": "{:.3f}",
                               "p-value": "{:.2e}", "% change in odds": "{:+.0f}%"}))

b_t = m4.params["np.log(tenure + 1)"]
lo_t, hi_t = ci4.loc["np.log(tenure + 1)"]
print(f"Doubling (tenure + 1) multiplies the odds of churn by {2**b_t:.3f}  (95% CI {2**lo_t:.3f} – {2**hi_t:.3f})")

,OR,CI 2.5%,CI 97.5%,p-value,% change in odds
Contract: Two year,0.191,0.133,0.275,6.10e-19,-81%
InternetService: No,0.436,0.320,0.594,1.38e-07,-56%
Contract: One year,0.462,0.366,0.584,1.09e-10,-54%
log(tenure + 1),0.488,0.446,0.534,1.19e-55,-51%
PhoneService: Yes,0.555,0.416,0.739,5.85e-05,-45%
OnlineSecurity: Yes,0.701,0.582,0.845,1.93e-04,-30%
TechSupport: Yes,0.750,0.622,0.904,2.55e-03,-25%
Dependents: Yes,0.818,0.682,0.981,2.99e-02,-18%
OnlineBackup: Yes,0.872,0.739,1.030,1.08e-01,-13%
PaymentMethod: Mailed check,1.000,0.772,1.296,9.98e-01,+0%


Doubling (tenure + 1) multiplies the odds of churn by 0.608  (95% CI 0.572 – 0.647)


In [50]:
# --- Graph 9: forest plot of odds ratios ---
ot = or_table.copy()
sig = (ot["CI 2.5%"] > 1) | (ot["CI 97.5%"] < 1)
colors = np.where(~sig, PAL["grey"], np.where(ot["OR"] > 1, PAL["churn"], PAL["stay"]))
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ot["OR"], y=ot.index, mode="markers",
    marker=dict(size=12, color=colors, symbol="diamond", line=dict(color="white", width=1)),
    error_x=dict(type="data", symmetric=False, array=ot["CI 97.5%"] - ot["OR"],
                 arrayminus=ot["OR"] - ot["CI 2.5%"], color="#64748B", thickness=2, width=0),
    customdata=np.stack([ot["CI 2.5%"], ot["CI 97.5%"], ot["p-value"]], axis=1),
    hovertemplate="<b>%{y}</b><br>OR = %{x:.2f}<br>95% CI %{customdata[0]:.2f} – %{customdata[1]:.2f}"
                  "<br>p = %{customdata[2]:.2e}<extra></extra>"))
fig.add_vline(x=1, line_dash="dot", line_color=PAL["dark"], annotation_text="OR = 1 (no effect)")
fig.add_annotation(x=np.log10(0.25), y=1.06, yref="paper", text="◀ lower churn risk", showarrow=False,
                   font=dict(color=PAL["stay"], size=13))
fig.add_annotation(x=np.log10(2.5), y=1.06, yref="paper", text="higher churn risk ▶", showarrow=False,
                   font=dict(color=PAL["churn"], size=13))
fig.update_layout(title="Adjusted odds ratios of churn with 95% CI (final model M4)",
                  xaxis=dict(type="log", title="Odds ratio (log scale)",
                             tickvals=[0.15, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4]),
                  showlegend=False, height=640, width=620, margin=dict(t=110))
fig.show()

## Step 10 — Benchmark: classification tree

A classification tree (Ch. 8) recursively splits the predictor space to create groups that are as homogeneous in churn as possible. It captures interactions and nonlinearities automatically, but it is less stable and does not give interpretable effect sizes.

To avoid overfitting, `max_depth` and `min_samples_leaf` are tuned by **5-fold stratified cross-validation on the training set** (criterion: ROC-AUC). The tree receives *all* variables, including the charges, because multicollinearity is not a problem for prediction.

In [51]:
X_all = pd.get_dummies(df[CATS + NUMS], drop_first=True).astype(float)
X_train, X_test = X_all.loc[train.index], X_all.loc[test.index]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid = GridSearchCV(DecisionTreeClassifier(random_state=SEED),
                    {"max_depth": [3, 4, 5, 6, 8], "min_samples_leaf": [20, 50, 100]},
                    scoring="roc_auc", cv=cv)
grid.fit(X_train, train["y"])
tree = grid.best_estimator_
print("Best parameters:", grid.best_params_, f"| CV AUC = {grid.best_score_:.3f} | leaves = {tree.get_n_leaves()}")
print("\nFirst levels of the tree:\n")
print(export_text(tree, feature_names=list(X_train.columns), max_depth=3))

Best parameters: {'max_depth': 5, 'min_samples_leaf': 50} | CV AUC = 0.832 | leaves = 30

First levels of the tree:

|--- tenure <= 16.50
|   |--- InternetService_Fiber optic <= 0.50
|   |   |--- tenure <= 3.50
|   |   |   |--- InternetService_No <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- InternetService_No >  0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |--- tenure >  3.50
|   |   |   |--- InternetService_No <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- InternetService_No >  0.50
|   |   |   |   |--- truncated branch of depth 2
|   |--- InternetService_Fiber optic >  0.50
|   |   |--- TotalCharges <= 120.00
|   |   |   |--- MonthlyCharges <= 70.58
|   |   |   |   |--- class: 1
|   |   |   |--- MonthlyCharges >  70.58
|   |   |   |   |--- truncated branch of depth 2
|   |   |--- TotalCharges >  120.00
|   |   |   |--- MonthlyCharges <= 80.58
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- Mont

In [52]:
# --- Graph 10: tree feature importance ---
imp = pd.Series(tree.feature_importances_, index=X_train.columns)
imp = imp[imp > 0.005].sort_values()
fig = go.Figure(go.Bar(x=imp.values, y=imp.index, orientation="h", marker_color=PAL["dark"],
                       text=[f"{v:.1%}" for v in imp.values], textposition="outside"))
fig.update_layout(title="Decision tree: tenure and fiber optic dominate (same story as the logistic model)",
                  xaxis=dict(title="Gini importance", tickformat=".0%"), height=480, width=620)
fig.show()

## Step 11 — Test-set comparison of all candidate models

All five models are evaluated on the **untouched test set**:

| Metric | Meaning |
|---|---|
| **ROC-AUC** | Probability that a random churner gets a higher score than a random non-churner (threshold-free ranking quality) |
| **Brier score** | Mean squared error of the predicted probabilities (calibration + discrimination; lower is better) |
| **Log-loss** | Negative mean log-likelihood on the test set (lower is better) |
| Accuracy / Precision / Recall / F1 | At the default 0.5 threshold (revisited in Step 12) |

To test whether the logistic model really beats the tree, we compute a **bootstrap 95% CI for the AUC difference** (2,000 resamples of the test set).

In [53]:
models = {"M1 Full": m1, "M2 Clean full": m2, "M3 Reduced": m3, "M4 Final": m4}
probs = {name: mod.predict(test).values for name, mod in models.items()}
probs["Decision tree"] = tree.predict_proba(X_test)[:, 1]

def evaluate(y, p_, thr=0.5):
    yhat = (p_ >= thr).astype(int)
    return {"AUC": roc_auc_score(y, p_), "Brier": brier_score_loss(y, p_), "Log-loss": log_loss(y, p_),
            "Accuracy": accuracy_score(y, yhat), "Precision": precision_score(y, yhat),
            "Recall": recall_score(y, yhat), "F1": f1_score(y, yhat)}

y_test = test["y"].values
comparison = pd.DataFrame({name: evaluate(y_test, p_) for name, p_ in probs.items()}).T
comparison.insert(0, "AIC (train)", [m1.aic, m2.aic, m3.aic, m4.aic, np.nan])
comparison.insert(0, "# params", [len(m.params) for m in models.values()] + [tree.get_n_leaves()])
display(comparison.style.format(precision=3).highlight_max(subset=["AUC"], color="#D1FAE5")
        .highlight_min(subset=["AIC (train)", "Brier", "Log-loss"], color="#D1FAE5"))

# Bootstrap CI for AUC(M4) - AUC(tree)
rng = np.random.default_rng(SEED)
diffs = []
for _ in range(2000):
    idx = rng.integers(0, len(y_test), len(y_test))
    diffs.append(roc_auc_score(y_test[idx], probs["M4 Final"][idx]) -
                 roc_auc_score(y_test[idx], probs["Decision tree"][idx]))
diffs = np.array(diffs)
lo_d, hi_d = np.percentile(diffs, [2.5, 97.5])
print(f"AUC(M4) − AUC(tree) = {diffs.mean():.4f}   95% bootstrap CI [{lo_d:.4f}, {hi_d:.4f}]")

,# params,AIC (train),AUC,Brier,Log-loss,Accuracy,Precision,Recall,F1
M1 Full,24,4675.454,0.835,0.141,0.429,0.797,0.632,0.564,0.596
M2 Clean full,22,4691.804,0.834,0.141,0.429,0.803,0.644,0.580,0.610
M3 Reduced,19,4686.316,0.834,0.141,0.429,0.803,0.644,0.580,0.610
M4 Final,19,4587.287,0.840,0.138,0.421,0.799,0.644,0.545,0.590
Decision tree,30,nan,0.822,0.145,0.464,0.779,0.587,0.567,0.577


AUC(M4) − AUC(tree) = 0.0184   95% bootstrap CI [0.0062, 0.0302]


In [54]:
# --- Graph 11a: ROC curves ---
palette = {"M1 Full": PAL["grey"], "M2 Clean full": PAL["gold"], "M3 Reduced": PAL["green"],
           "M4 Final": PAL["churn"], "Decision tree": PAL["dark"]}
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random (AUC = 0.5)",
                         line=dict(color="#CBD5E1", dash="dot")))
for name, p_ in probs.items():
    fpr, tpr, _ = roc_curve(y_test, p_)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{name} ({roc_auc_score(y_test, p_):.3f})",
                             line=dict(color=palette[name], width=4 if name == "M4 Final" else 2)))
fig.update_layout(title="ROC curves on the test set (AUC in parentheses)",
                  xaxis_title="False positive rate (1 − specificity)", yaxis_title="True positive rate (recall)",
                  height=520, width=620, legend=dict(orientation="v", x=0.55, y=0.05, yanchor="bottom"))
fig.show()

# --- Graph 11b: bootstrap distribution of the AUC difference ---
fig = go.Figure(go.Histogram(x=diffs, nbinsx=50, marker_color=PAL["stay"], opacity=0.85))
fig.add_vline(x=0, line_color=PAL["dark"], line_dash="dot", annotation_text="no difference")
fig.add_vrect(x0=lo_d, x1=hi_d, fillcolor=PAL["green"], opacity=0.15, line_width=0, annotation_text="95% CI")
fig.update_layout(title="Bootstrap: AUC(M4) − AUC(tree). The CI excludes 0, so logistic regression is better",
                  xaxis_title="AUC difference", yaxis_title="Bootstrap samples", height=400, width=620)
fig.show()

## Step 12 — Class imbalance and the choice of threshold

With 73% non-churners, a model that predicts "no churn" for everyone already reaches **73% accuracy**, so accuracy is a poor guide. The default cut-off of 0.5 also favours the majority class and misses many churners.

**Procedure (no test-set snooping):**
1. Obtain **out-of-fold** predicted probabilities on the training set with 5-fold stratified CV.
2. Choose the threshold that maximizes **F1** (balance of precision and recall). For reference, also compute the **Youden** threshold (maximizes TPR − FPR).
3. Apply the chosen threshold **once** to the test set.

**Business view:** a false negative (a churner we miss) usually costs much more than a false positive (an offer sent to someone who would have stayed). If the costs are known, the optimal threshold is roughly
$t^* = \dfrac{\text{cost of the offer}}{\text{value of a saved customer} \times \text{offer success rate}}$.

In [55]:
oof = np.zeros(len(train))
for tr_idx, va_idx in cv.split(train, train["y"]):
    mod = fit_logit(FINAL_TERMS, data=train.iloc[tr_idx])
    oof[va_idx] = mod.predict(train.iloc[va_idx]).values

thresholds = np.round(np.arange(0.05, 0.81, 0.01), 2)
curve = pd.DataFrame({
    "threshold": thresholds,
    "Precision": [precision_score(train["y"], oof >= t, zero_division=0) for t in thresholds],
    "Recall": [recall_score(train["y"], oof >= t) for t in thresholds],
    "F1": [f1_score(train["y"], oof >= t) for t in thresholds],
})
T_F1 = float(curve.loc[curve["F1"].idxmax(), "threshold"])
fpr_cv, tpr_cv, thr_cv = roc_curve(train["y"], oof)
T_YOUDEN = float(thr_cv[np.argmax(tpr_cv - fpr_cv)])
print(f"CV AUC (train, out-of-fold) = {roc_auc_score(train['y'], oof):.3f}")
print(f"F1-optimal threshold = {T_F1:.2f} | Youden threshold = {T_YOUDEN:.2f} | base rate = {train['y'].mean():.2f}")

p4 = probs["M4 Final"]
rows = []
for name_t, t_ in [("Default 0.50", 0.50), (f"F1-optimal {T_F1:.2f}", T_F1), (f"Youden {T_YOUDEN:.2f}", T_YOUDEN)]:
    tn, fp, fn, tp = confusion_matrix(y_test, p4 >= t_).ravel()
    rows.append({"threshold": name_t, "TN": tn, "FP": fp, "FN": fn, "TP": tp, "flagged": tp + fp,
                 **{k_: v_ for k_, v_ in evaluate(y_test, p4, t_).items() if k_ in ["Accuracy", "Precision", "Recall", "F1"]}})
thr_table = pd.DataFrame(rows).set_index("threshold")
display(thr_table.style.format({"Accuracy": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}"}))

CV AUC (train, out-of-fold) = 0.852
F1-optimal threshold = 0.29 | Youden threshold = 0.25 | base rate = 0.27


,TN,FP,FN,TP,flagged,Accuracy,Precision,Recall,F1
threshold,,,,,,,,,
Default 0.50,920,113,170,204,317,0.799,0.644,0.545,0.590
F1-optimal 0.29,759,274,85,289,563,0.745,0.513,0.773,0.617
Youden 0.25,709,324,69,305,629,0.721,0.485,0.816,0.608


In [56]:
# --- Graph 12a: precision / recall / F1 vs threshold (CV on train) ---
fig = go.Figure()
for metric, col in [("Precision", PAL["stay"]), ("Recall", PAL["churn"]), ("F1", PAL["dark"])]:
    fig.add_trace(go.Scatter(x=curve["threshold"], y=curve[metric], mode="lines", name=metric,
                             line=dict(color=col, width=3 if metric == "F1" else 2)))
fig.add_vline(x=0.5, line_dash="dot", line_color=PAL["grey"], annotation_text="default 0.50")
fig.add_vline(x=T_F1, line_dash="dash", line_color=PAL["green"], annotation_text=f"chosen {T_F1:.2f}",
              annotation_position="top left")
fig.update_layout(title="Threshold trade-off (out-of-fold predictions on the training set)",
                  xaxis_title="Classification threshold", yaxis=dict(tickformat=".0%", range=[0, 1.02]), height=440, width=620)
fig.show()

# --- Graph 12b: confusion matrices on the test set ---
fig = make_subplots(rows=1, cols=2, subplot_titles=["Threshold 0.50", f"Threshold {T_F1:.2f} (chosen)"],
                    horizontal_spacing=0.12)
for i, t_ in enumerate([0.50, T_F1], start=1):
    cm = confusion_matrix(y_test, p4 >= t_)
    pct = cm / cm.sum(axis=1, keepdims=True)
    text = [[f"<b>{cm[r, c_]}</b><br>{pct[r, c_]:.0%} of row" for c_ in range(2)] for r in range(2)]
    fig.add_trace(go.Heatmap(z=pct, x=["Pred: Stay", "Pred: Churn"], y=["Actual: Stay", "Actual: Churn"],
                             text=text, texttemplate="%{text}", textfont=dict(size=14),
                             colorscale=[[0, "#F8FAFC"], [1, PAL["stay"]]], showscale=False, zmin=0, zmax=1),
                  row=1, col=i)
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Test-set confusion matrices: the lower threshold catches far more churners", height=420, width=620)
fig.show()

## Step 13 — Gains and lift: how well does the model rank customers?

A retention team rarely contacts everyone. The useful question is: *if we contact the top x% riskiest customers, what share of all churners do we reach?*

- **Cumulative gains curve:** % of churners captured vs % of customers contacted (diagonal = random targeting).
- **Lift by decile:** churn rate in each risk decile divided by the overall churn rate.

In [57]:
g = pd.DataFrame({"y": y_test, "p": p4}).sort_values("p", ascending=False).reset_index(drop=True)
g["decile"] = pd.qcut(g.index, 10, labels=range(1, 11))
dec = g.groupby("decile", observed=True).agg(churn_rate=("y", "mean"), churners=("y", "sum"), n=("y", "size"))
dec["lift"] = dec["churn_rate"] / g["y"].mean()
dec["cum % churners captured"] = dec["churners"].cumsum() / g["y"].sum()
display(dec.style.format({"churn_rate": "{:.1%}", "lift": "{:.2f}", "cum % churners captured": "{:.1%}"}))

g["cum_customers"] = (g.index + 1) / len(g)
g["cum_churners"] = g["y"].cumsum() / g["y"].sum()

,churn_rate,churners,n,lift,cum % churners captured
decile,,,,,
1,74.5%,105,141,2.80,28.1%
2,56.0%,79,141,2.11,49.2%
3,45.7%,64,140,1.72,66.3%
4,29.1%,41,141,1.09,77.3%
5,23.4%,33,141,0.88,86.1%
6,20.0%,28,140,0.75,93.6%
7,9.2%,13,141,0.35,97.1%
8,6.4%,9,140,0.24,99.5%
9,0.7%,1,141,0.03,99.7%


In [58]:
# --- Graph 13: gains curve + lift by decile ---
fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative gains", "Churn rate by risk decile"],
                    horizontal_spacing=0.1)
fig.add_trace(go.Scatter(x=g["cum_customers"], y=g["cum_churners"], mode="lines", name="M4",
                         line=dict(color=PAL["churn"], width=3), fill="tonexty"), row=1, col=1)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random",
                         line=dict(color=PAL["grey"], dash="dot")), row=1, col=1)
at30 = g.loc[g["cum_customers"] <= 0.30, "cum_churners"].max()
fig.add_annotation(x=0.30, y=at30, text=f"Top 30% → {at30:.0%} of churners", showarrow=True,
                   arrowhead=2, ax=60, ay=40, row=1, col=1)
fig.add_trace(go.Bar(x=dec.index.astype(str), y=dec["churn_rate"], name="Churn rate",
                     marker_color=[PAL["churn"] if r > g["y"].mean() else PAL["stay"] for r in dec["churn_rate"]],
                     text=[f"{r:.0%}" for r in dec["churn_rate"]], textposition="outside"), row=1, col=2)
fig.add_hline(y=g["y"].mean(), line_dash="dot", line_color=PAL["dark"], row=1, col=2)
fig.update_xaxes(title_text="% of customers contacted (highest risk first)", tickformat=".0%", row=1, col=1)
fig.update_yaxes(title_text="% of churners captured", tickformat=".0%", row=1, col=1)
fig.update_xaxes(title_text="Risk decile (1 = highest)", row=1, col=2)
fig.update_yaxes(tickformat=".0%", range=[0, 0.9], row=1, col=2)
fig.update_layout(title="Targeting efficiency of the final model on the test set", showlegend=False, height=460, width=620)
fig.show()

## Step 14 — Recommendations: which customers to target and why

We now use the final model to translate odds ratios into **actionable segments**:

1. **Risk map:** observed churn by tenure band × contract, split by internet service.
2. **Priority segments:** size, churn rate and share of all churners.
3. **What-if scenarios:** predicted churn probability of a typical high-risk customer when one characteristic changes. This shows *which levers* the model associates with the largest reductions in risk. These are associations, so the real effect of an offer should be validated with an A/B test.
4. **Revenue at risk** in the highest-risk group.

For profiling we score the whole dataset with M4 (the model was estimated only on the training set).

In [59]:
df["p_churn"] = m4.predict(df).values
df["tenure_band"] = pd.cut(df["tenure"], [-1, 6, 12, 24, 48, 72],
                           labels=["0–6", "7–12", "13–24", "25–48", "49–72"])

segments = {
    "All customers": df["y"] >= 0,
    "Month-to-month + Fiber": (df.Contract == "Month-to-month") & (df.InternetService == "Fiber optic"),
    "Month-to-month + Fiber + tenure ≤ 12":
        (df.Contract == "Month-to-month") & (df.InternetService == "Fiber optic") & (df.tenure <= 12),
    "… + paying by electronic check":
        (df.Contract == "Month-to-month") & (df.InternetService == "Fiber optic") & (df.tenure <= 12)
        & (df.PaymentMethod == "Electronic check"),
    "Month-to-month + Fiber, no security & no tech support":
        (df.Contract == "Month-to-month") & (df.InternetService == "Fiber optic")
        & (df.OnlineSecurity == "No") & (df.TechSupport == "No"),
    "Two-year contract": df.Contract == "Two year",
}
seg_table = pd.DataFrame({
    name: {"customers": mask.sum(), "% of base": mask.mean(), "churn rate": df.loc[mask, "y"].mean(),
           "% of all churners": df.loc[mask, "y"].sum() / df["y"].sum(),
           "avg monthly bill": df.loc[mask, "MonthlyCharges"].mean()}
    for name, mask in segments.items()}).T
display(seg_table.style.format({"customers": "{:,.0f}", "% of base": "{:.1%}", "churn rate": "{:.1%}",
                                "% of all churners": "{:.1%}", "avg monthly bill": "${:.2f}"}))

# Revenue at risk in the top 30% predicted risk
top = df["p_churn"] >= df["p_churn"].quantile(0.70)
rev_top = df.loc[top & (df.y == 1), "MonthlyCharges"].sum()
rev_all = df.loc[df.y == 1, "MonthlyCharges"].sum()
print(f"Top 30% predicted risk: churn rate {df.loc[top, 'y'].mean():.1%}, "
      f"captures {df.loc[top, 'y'].sum() / df.y.sum():.1%} of churners and "
      f"${rev_top:,.0f} of ${rev_all:,.0f} monthly revenue lost ({rev_top / rev_all:.0%})")

,customers,% of base,churn rate,% of all churners,avg monthly bill
All customers,"7,032",100.0%,26.6%,100.0%,$64.80
Month-to-month + Fiber,"2,128",30.3%,54.6%,62.2%,$87.02
Month-to-month + Fiber + tenure ≤ 12,916,13.0%,70.2%,34.4%,$82.08
… + paying by electronic check,631,9.0%,71.2%,24.0%,$82.31
"Month-to-month + Fiber, no security & no tech support","1,524",21.7%,60.7%,49.5%,$84.65
Two-year contract,"1,685",24.0%,2.8%,2.6%,$60.87


Top 30% predicted risk: churn rate 59.8%, captures 67.5% of churners and $99,105 of $139,131 monthly revenue lost (71%)


In [63]:
order_contract = ["Month-to-month", "One year", "Two year"]

fig = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=["DSL", "Fiber optic", "No internet"],
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.08
)

for i, net in enumerate(["DSL", "Fiber optic", "No"], start=1):

    sub = df[df["InternetService"] == net]

    piv = sub.pivot_table(
        index="Contract",
        columns="tenure_band",
        values="y",
        aggfunc="mean",
        observed=False
    )

    cnt = sub.pivot_table(
        index="Contract",
        columns="tenure_band",
        values="y",
        aggfunc="size",
        observed=False
    )

    piv = piv.reindex(order_contract)
    cnt = cnt.reindex(order_contract)

    text = [
        [
            (
                f"{piv.iloc[r, c_]:.0%}<br>n={int(cnt.iloc[r, c_])}"
                if pd.notna(piv.iloc[r, c_])
                else ""
            )
            for c_ in range(piv.shape[1])
        ]
        for r in range(piv.shape[0])
    ]

    fig.add_trace(
        go.Heatmap(
            z=piv.values,
            x=[str(c_) for c_ in piv.columns],
            y=piv.index,
            text=text,
            texttemplate="%{text}",
            zmin=0,
            zmax=0.8,
            showscale=(i == 3),
            colorscale=[
                [0, "#F0F9FF"],
                [0.35, PAL["gold"]],
                [1, PAL["churn"]]
            ],
            colorbar=dict(
                title="Churn",
                tickformat=".0%"
            )
        ),
        row=i,
        col=1
    )

# X axis
fig.update_xaxes(
    title_text="Tenure (months)",
    row=3,
    col=1
)

# Y axis
fig.update_yaxes(
    autorange="reversed"
)

fig.update_layout(
    title="Churn Rate by Contract, Tenure, and Internet Service",
    width=620,
    height=850
)

fig.show()

In [64]:
# --- What-if scenarios for a typical high-risk customer ---
prototype = dict(gender="Female", SeniorCitizen="No", Partner="No", Dependents="No", PhoneService="Yes",
                 MultipleLines="No", InternetService="Fiber optic", OnlineSecurity="No", OnlineBackup="No",
                 DeviceProtection="No", TechSupport="No", StreamingTV="No", StreamingMovies="No",
                 Contract="Month-to-month", PaperlessBilling="Yes", PaymentMethod="Electronic check", tenure=3)

def predict_one(**changes):
    return float(m4.predict(pd.DataFrame([{**prototype, **changes}])).iloc[0])

scenarios = {
    "Baseline: 3 months, fiber, month-to-month, e-check, no add-ons": {},
    "Switches to automatic credit-card payment": {"PaymentMethod": "Credit card (automatic)"},
    "Adds online security + tech support": {"OnlineSecurity": "Yes", "TechSupport": "Yes"},
    "Signs a one-year contract": {"Contract": "One year"},
    "Same customer at 12 months": {"tenure": 12},
    "Same customer at 24 months": {"tenure": 24},
    "Signs a two-year contract": {"Contract": "Two year"},
}
what_if = pd.Series({k_: predict_one(**v_) for k_, v_ in scenarios.items()}, name="P(churn)")
display(what_if.to_frame().style.format("{:.1%}"))

,P(churn)
"Baseline: 3 months, fiber, month-to-month, e-check, no add-ons",70.9%
Switches to automatic credit-card payment,64.7%
Adds online security + tech support,56.1%
Signs a one-year contract,53.0%
Same customer at 12 months,51.1%
Same customer at 24 months,39.5%
Signs a two-year contract,31.8%


In [65]:
# --- Graph 14b: what-if scenarios ---
w = what_if.sort_values()
base_p = what_if.iloc[0]
fig = go.Figure(go.Bar(x=w.values, y=w.index, orientation="h",
                       marker_color=[PAL["churn"] if v_ == base_p else PAL["stay"] for v_ in w.values],
                       text=[f"{v_:.0%}" + ("" if v_ == base_p else f"  ({(v_ - base_p) * 100:+.0f} pts)") for v_ in w.values],
                       textposition="outside"))
fig.add_vline(x=base_p, line_dash="dot", line_color=PAL["churn"])
fig.update_layout(title="Predicted churn for a typical high-risk customer under different scenarios",
                  xaxis=dict(tickformat=".0%", range=[0, 0.9], title="Predicted probability of churn"),
                  height=440, width=620)
fig.show()

### Recommended retention targets

| Priority | Who | Why (evidence from the model) | Suggested action |
|---|---|---|---|
| **1** | **New (≤ 12 months), month-to-month, fiber-optic customers**, especially those paying by electronic check | Short tenure, no contract and fiber are the three strongest risk factors, and they combine multiplicatively on the odds scale. This segment has a ~70% churn rate and contains about a third of all churners. | Offer a discount for a 1- or 2-year contract during the first months, plus a proactive onboarding call |
| **2** | **Month-to-month fiber customers without online security or tech support** | Both add-ons are protective (OR ≈ 0.70 and 0.75); this group contains about half of all churners | Bundle security + tech support (free trial) |
| **3** | **Electronic-check payers** | OR ≈ 1.39 vs bank transfer, after adjusting for everything else | Incentive to move to automatic payment |
| — | **Fiber optic overall** | OR ≈ 2.6 vs DSL: price or service-quality problem | Investigate pricing and network quality (a product issue, not just retention) |
| Low | Two-year contract, long-tenure customers | Churn under 3% | No proactive spending needed |

Using the model's ranking with the **0.29 threshold** (or simply contacting the **top 30% riskiest customers**) focuses the budget where most churners and most of the lost revenue are.

## Step 15 — Conclusions

**Model selection**
- The full model (M1) suffered from severe structural multicollinearity: `MonthlyCharges` is a deterministic function of the services (R² ≈ 0.999) and `TotalCharges` ≈ tenure × MonthlyCharges. Both were removed.
- Backward stepwise AIC removed `Partner`, `gender` and `DeviceProtection`; the LRT against the clean full model was not significant, so the reduced model is preferred.
- The logit is **not linear** in tenure (empirical logits, Box-Tidwell). Using `log(tenure + 1)` lowered AIC by about 100 points.
- The final model **M4** has the lowest AIC, the best test-set AUC, Brier score and log-loss, and it significantly outperforms the decision tree (bootstrap CI of the AUC difference excludes 0).

**Assumptions**
- After the fix, all VIFs are below 3.
- No influential observations (max Cook's D ≈ 0.003); deleting "influential" points would induce separation.
- Hosmer-Lemeshow shows no lack of fit; the calibration curve follows the diagonal.

**Main findings (adjusted odds ratios)**
- Contract is the strongest protective factor: two-year contracts have about **81% lower odds** of churn than month-to-month.
- Each doubling of tenure multiplies the odds by about **0.61**.
- Fiber optic customers have about **2.6 times** the odds of DSL customers.
- Electronic check, paperless billing, streaming services and multiple lines increase risk; online security, tech support and having dependents decrease it.

**Classification**
- Test AUC ≈ 0.84. Because of the 73/27 imbalance, the default 0.5 threshold misses almost half of the churners. The CV-selected threshold (≈ 0.29, close to the base rate) raises recall to about 77% at the cost of lower precision, which is a sensible trade-off for retention campaigns.

**Limitations and next steps**
- Observational data: odds ratios are associations, not causal effects. Validate retention offers with randomized experiments (A/B tests).
- Price effects are absorbed by the service variables after removing `MonthlyCharges`.
- Cross-sectional snapshot: a survival model (time-to-churn) would use tenure more naturally.
- Interactions (e.g. Contract × InternetService) were not explored and are a natural extension.